In [ ]:
!pip install  transformers datasets peft torch>0.16.0 pandas --quiet

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset as HFDataset
from peft import LoraConfig, get_peft_model, TaskType

torch.manual_seed(42)


## Load Data



In [ ]:
TRAIN_CSV_PATH = "/content/train.csv"

train_df = pd.read_csv(TRAIN_CSV_PATH)
train_df.head()


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


## Q1. Label Encoding


In [ ]:
label_map = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

train_df["label"] = train_df["answer"].map(label_map)

q1_answer = train_df.loc[150, "label"]
print("Q1 - Encoded label at index 150:", q1_answer)


Q1 - Encoded label at index 150: 2


In [ ]:
row0 = train_df.loc[0]

option_b_input = str(row0["prompt"]) + " [SEP] " + str(row0["B"])

q2_answer = len(option_b_input)
print("Formatted Option B input:")
print(option_b_input)
print()
print("Q2 - Character length:", q2_answer)


Formatted Option B input:
Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.

Q2 - Character length: 407


In [ ]:
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LENGTH = 128
options_cols = ["A", "B", "C", "D", "E"]

def build_choices(row):
    return [str(row["prompt"]) + " [SEP] " + str(row[opt]) for opt in options_cols]

row0_choices = build_choices(row0)

encoded = tokenizer(
    row0_choices,
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
)

# Reshape to [batch_size, num_choices, sequence_length]
input_ids = encoded["input_ids"].unsqueeze(0)   # [1, 5, 128]
attention_mask = encoded["attention_mask"].unsqueeze(0)

print("input_ids shape:", input_ids.shape)
q3_answer = input_ids.shape[1]
print("Q3 - Second dimension (num_choices):", q3_answer)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

input_ids shape: torch.Size([1, 5, 128])
Q3 - Second dimension (num_choices): 5


In [ ]:
N_ROWS = 16
batch_rows = train_df.iloc[:N_ROWS]

all_choices = []
for _, r in batch_rows.iterrows():
    all_choices.extend(build_choices(r))

batch_encoded = tokenizer(
    all_choices,
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
)

batch_input_ids = batch_encoded["input_ids"].view(N_ROWS, 5, MAX_LENGTH)
print("Batch input_ids shape:", batch_input_ids.shape)

q4_answer = batch_input_ids.numel()
print("Q4 - Total token positions:", q4_answer)


Batch input_ids shape: torch.Size([16, 5, 128])
Q4 - Total token positions: 10240


In [ ]:
mc_model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
mc_model.eval()

# input_ids / attention_mask from Q3 already shaped [1, 5, 128]
with torch.no_grad():
    outputs = mc_model(input_ids=input_ids, attention_mask=attention_mask)

logits = outputs.logits
print("Logits shape:", logits.shape)

q5_answer = logits.shape[1]
print("Q5 - Number of logits for one question:", q5_answer)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Logits shape: torch.Size([1, 5])
Q5 - Number of logits for one question: 5


In [ ]:
label_row0 = torch.tensor([train_df.loc[0, "label"]])

with torch.no_grad():
    outputs_with_loss = mc_model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=label_row0,
    )

loss = outputs_with_loss.loss
print("Loss value:", loss.item())
print("Loss tensor shape:", loss.shape)

q6_answer = loss.dim()
print("Q6 - Number of dimensions in loss tensor:", q6_answer)


Loss value: 1.6243171691894531
Loss tensor shape: torch.Size([])
Q6 - Number of dimensions in loss tensor: 0


In [ ]:
!pip install --upgrade torchao


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 37.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

lora_model = get_peft_model(mc_model, lora_config)

q7_answer = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
print("Q7 - Trainable parameters:", q7_answer)

lora_model.print_trainable_parameters()


Q7 - Trainable parameters: 295681
trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693


In [ ]:
N_HF = 100
hf_rows = train_df.iloc[:N_HF].reset_index(drop=True)

def encode_row(row):
    choices = build_choices(row)
    enc = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    return {
        "input_ids": enc["input_ids"],            
        "attention_mask": enc["attention_mask"],  
        "labels": row["label"],
    }

records = [encode_row(r) for _, r in hf_rows.iterrows()]

hf_dataset = HFDataset.from_dict({
    "input_ids": [rec["input_ids"].tolist() for rec in records],
    "attention_mask": [rec["attention_mask"].tolist() for rec in records],
    "labels": [rec["labels"] for rec in records],
})

first_item_input_ids_shape = np.array(hf_dataset[0]["input_ids"]).shape
print("First dataset item input_ids shape:", first_item_input_ids_shape)

q8_answer = first_item_input_ids_shape[0]
print("Q8 - Number of tokenized choices stored in input_ids:", q8_answer)


First dataset item input_ids shape: (5, 128)
Q8 - Number of tokenized choices stored in input_ids: 5


In [ ]:
N_TRAIN = 32
MAX_LEN_TINY = 64

tiny_rows = train_df.iloc[:N_TRAIN].reset_index(drop=True)

def encode_row_tiny(row):
    choices = build_choices(row)
    enc = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN_TINY,
        return_tensors="pt",
    )
    return {
        "input_ids": enc["input_ids"].tolist(),
        "attention_mask": enc["attention_mask"].tolist(),
        "labels": int(row["label"]),
    }

tiny_records = [encode_row_tiny(r) for _, r in tiny_rows.iterrows()]

tiny_dataset = HFDataset.from_dict({
    "input_ids": [r["input_ids"] for r in tiny_records],
    "attention_mask": [r["attention_mask"] for r in tiny_records],
    "labels": [r["labels"] for r in tiny_records],
})

class MCDataCollator:
    def __call__(self, features):
        batch = {}
        batch["input_ids"] = torch.tensor([f["input_ids"] for f in features])
        batch["attention_mask"] = torch.tensor([f["attention_mask"] for f in features])
        batch["labels"] = torch.tensor([f["labels"] for f in features])
        return batch

data_collator = MCDataCollator()

# Fresh base model + LoRA for fine-tuning
base_model_for_ft = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
ft_lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)
ft_model = get_peft_model(base_model_for_ft, ft_lora_config)

training_args = TrainingArguments(
    output_dir="./mcq_lora_tmp",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    save_strategy="no",
    report_to=[],
)

trainer = Trainer(
    model=ft_model,
    args=training_args,
    train_dataset=tiny_dataset,
    data_collator=data_collator,
)

train_result = trainer.train()

q9_answer = trainer.state.global_step
print("Q9 - Final global_step:", q9_answer)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist

Step,Training Loss
1,1.599842
2,1.628842
3,1.569262
4,1.603194


Q9 - Final global_step: 4


In [ ]:
ft_model.eval()

row0_choices_tiny = build_choices(row0)
encoded_row0_tiny = tokenizer(
    row0_choices_tiny,
    padding="max_length",
    truncation=True,
    max_length=MAX_LEN_TINY,
    return_tensors="pt",
)

infer_input_ids = encoded_row0_tiny["input_ids"].unsqueeze(0)
infer_attention_mask = encoded_row0_tiny["attention_mask"].unsqueeze(0)
# device = next(ft_model.parameters()).device
with torch.no_grad():
    infer_outputs = ft_model(
        input_ids=infer_input_ids,
        attention_mask=infer_attention_mask,
    )

probs = torch.softmax(infer_outputs.logits, dim=-1).squeeze(0)
print("Probabilities [A, B, C, D, E]:", probs.tolist())

q10_answer = round(probs[4].item(), 4)
print("Q10 - Probability assigned to Option E:", q10_answer)


Probabilities [A, B, C, D, E]: [0.20490434765815735, 0.206613227725029, 0.20302529633045197, 0.19438128173351288, 0.1910758912563324]
Q10 - Probability assigned to Option E: 0.1911
